# Исследовательский анализ Geo Reviews Dataset 2023

**Цель:** проверить структуру и качество полного исходного набора из русскоязычных отзывов и оценить постановку бинарной классификации тональности.

Предварительное правило меток: рейтинги 1–2 — отрицательный класс, 4–5 — положительный, 0 и 3 — отдельно. Рейтинг используется только для получения черновой целевой метки и **не должен быть входным признаком модели**. Ноутбук ничего не удаляет и не изменяет в `data/raw`.

Публичная версия ноутбука хранится без outputs. Агрегированные графики сохраняются в `reports/figures/`; выполненная копия создаётся локально в `.local/executed_notebooks/`.

## 1. Среда выполнения и расположение данных

Пути определяются относительно корня проекта, поэтому ноутбук можно запускать как из корня, так и из каталога `notebooks`.

In [ ]:
from pathlib import Path
from time import perf_counter
import platform
import re

import numpy as np
import pandas as pd
import pyarrow
import matplotlib.pyplot as plt
from IPython.display import display

STARTED = perf_counter()
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (WORKDIR, *WORKDIR.parents)
                     if (p / "data" / "raw").is_dir()), None)
assert PROJECT_ROOT is not None, "Не найден корень проекта с data/raw"
GEO_ROOT = PROJECT_ROOT / "data" / "raw" / "geo_reviews"
parquet_files = sorted(GEO_ROOT.rglob("*.parquet"))
assert len(parquet_files) == 1, f"Ожидался один Parquet-файл, найдено: {len(parquet_files)}"
PARQUET_PATH = parquet_files[0]
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| pyarrow:", pyarrow.__version__)
print("Файл:", PARQUET_PATH.relative_to(PROJECT_ROOT))
print("Размер файла, байт:", PARQUET_PATH.stat().st_size)

## 2. Структура и типы данных

In [ ]:
read_started = perf_counter()
df = pd.read_parquet(PARQUET_PATH)
SOURCE_COLUMNS = df.columns.tolist()
print(f"Прочитано за {perf_counter() - read_started:.2f} с")
print("Размер таблицы:", df.shape)
print("Столбцы:", df.columns.tolist())
print("Типы данных:")
display(df.dtypes.rename("dtype").to_frame())
print("Примеры реальных записей:")
display(df.head(3))

In [ ]:
memory = df.memory_usage(deep=True).rename("байт").to_frame()
memory["МиБ"] = memory["байт"] / 1024**2
display(memory.round(2))
print(f"Всего в памяти: {memory['байт'].sum():,} байт ({memory['МиБ'].sum():.2f} МиБ)")

print("Фактические Python-типы первых непустых значений:")
for column in df.columns:
    non_null = df[column].dropna()
    print(f"  {column}: {type(non_null.iloc[0]).__name__ if len(non_null) else 'нет значений'}")
print("rubrics — строка с категориями через ';', а не вложенный список:")
display(df.loc[df["rubrics"].str.contains(";", regex=False), ["rubrics"]].head(3))

## 3. Оценка полноты и корректности строковых данных

Для каждой проблемы показываются и количество, и процент от всех строк. Пропуск (`NaN`) и пустая строка — разные случаи.

In [ ]:
quality_rows = []
for column in df.columns:
    missing = int(df[column].isna().sum())
    blank = 0
    if pd.api.types.is_string_dtype(df[column]):
        blank = int(df[column].fillna("").str.strip().eq("").sum() - missing)
    quality_rows.append({"столбец": column, "пропусков": missing,
                         "% пропусков": missing / len(df) * 100,
                         "пустых/пробельных строк": blank,
                         "% пустых": blank / len(df) * 100})
quality_df = pd.DataFrame(quality_rows).set_index("столбец")
display(quality_df.round(4))

text = df["text"].fillna("").astype(str)
encoding_checks = pd.Series({
    "строк с символом замены U+FFFD": text.str.contains("�", regex=False).sum(),
    "строк с нулевым байтом": text.str.contains("\x00", regex=False).sum(),
}).astype(int)
display(pd.DataFrame({"количество": encoding_checks,
                      "% строк": encoding_checks / len(df) * 100}).round(5))
if encoding_checks.iloc[0]:
    print("Пример с U+FFFD (текст сокращён):")
    display(df.loc[text.str.contains("�", regex=False), ["rating", "text"]]
              .assign(text=lambda x: x["text"].str.slice(0, 300)))

## 4. Анализ дубликатов

In [ ]:
full_duplicate_rows = int(df.duplicated(keep=False).sum())
full_duplicate_extra = int(df.duplicated().sum())
repeated_text_rows = int(df.duplicated("text", keep=False).sum())
repeated_text_extra = int(df.duplicated("text").sum())
repeated_text_values = int(df.loc[df.duplicated("text", keep=False), "text"].nunique())
duplicate_summary = pd.DataFrame({
    "проверка": ["полные дубликаты: все строки в группах", "полные дубликаты: лишние копии",
                 "повторы текста: все строки в группах", "повторы текста: лишние копии",
                 "уникальные тексты, которые повторяются"],
    "количество": [full_duplicate_rows, full_duplicate_extra, repeated_text_rows,
                   repeated_text_extra, repeated_text_values],
})
duplicate_summary["% от строк"] = duplicate_summary["количество"] / len(df) * 100
display(duplicate_summary.round(5))
print("Примеры повторяющихся текстов (это не всегда полные дубликаты):")
display(df.loc[df.duplicated("text", keep=False), ["rating", "name_ru", "rubrics", "text"]]
          .sort_values("text").head(10))

## 5. Распределение рейтингов и предварительная целевая схема

In [ ]:
expected_ratings = {0, 1, 2, 3, 4, 5}
actual_ratings = set(df["rating"].dropna().unique())
print("Фактические значения rating:", sorted(actual_ratings))
print("Неожиданные значения:", sorted(actual_ratings - expected_ratings))

rating_table = df["rating"].value_counts().sort_index().rename("количество").to_frame()
rating_table["процент"] = rating_table["количество"] / len(df) * 100
display(rating_table.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(rating_table.index.astype(str), rating_table["количество"], color="#4C78A8")
ax.bar_label(bars, labels=[f"{x:,}" for x in rating_table["количество"]], padding=3, fontsize=8)
ax.set(title="Распределение рейтингов", xlabel="Рейтинг", ylabel="Количество отзывов")
ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "geo_rating_distribution.png", dpi=150)
plt.show()

In [ ]:
negative_mask = df["rating"].isin([1, 2])
positive_mask = df["rating"].isin([4, 5])
binary_mask = negative_mask | positive_mask
binary_counts = pd.Series({"положительные (4–5)": int(positive_mask.sum()),
                           "отрицательные (1–2)": int(negative_mask.sum())})
binary_table = binary_counts.rename("количество").to_frame()
binary_table["% внутри бинарной выборки"] = binary_table["количество"] / binary_mask.sum() * 100
display(binary_table.round(3))
excluded = int((~binary_mask).sum())
print(f"Исключались бы рейтинги 0 и 3: {excluded:,} ({excluded / len(df) * 100:.3f}%)")
print(f"Отношение положительного класса к отрицательному: {positive_mask.sum() / negative_mask.sum():.2f}:1")
print("Вывод: схема понятна, но без балансировки рабочей выборки большинство класса 4–5 будет доминировать.")

## 6. Распределение длины текстов

In [ ]:
df["char_len"] = text.str.len()
df["word_len"] = text.str.count(r"\S+")
length_stats = df[["char_len", "word_len"]].describe(
    percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).T
display(length_stats.round(2))

thresholds = pd.Series({
    "не более 3 символов": df["char_len"].le(3).sum(),
    "менее 10 символов": df["char_len"].lt(10).sum(),
    "более 1 000 символов": df["char_len"].gt(1000).sum(),
    "более 2 000 символов": df["char_len"].gt(2000).sum(),
}).astype(int)
display(pd.DataFrame({"количество": thresholds,
                      "% строк": thresholds / len(df) * 100}).round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["char_len"].clip(upper=1500), bins=60, color="#59A14F")
axes[0].set(title="Длина отзыва в символах\n(1500+ объединены)", xlabel="Символов", ylabel="Отзывов")
axes[1].hist(df["word_len"].clip(upper=250), bins=50, color="#F28E2B")
axes[1].set(title="Длина отзыва в словах\n(250+ объединены)", xlabel="Слов", ylabel="Отзывов")
for ax in axes: ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "geo_text_lengths.png", dpi=150)
plt.show()

## 7. Сравнение групп предварительной тональности

In [ ]:
binary_df = df.loc[binary_mask, ["rating", "char_len", "word_len"]].copy()
binary_df["класс"] = np.where(binary_df["rating"].isin([4, 5]), "положительный", "отрицательный")
group_lengths = binary_df.groupby("класс")[["char_len", "word_len"]].describe(
    percentiles=[.05, .25, .50, .75, .95, .99])
display(group_lengths.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, color in [("положительный", "#59A14F"), ("отрицательный", "#E15759")]:
    subset = binary_df.loc[binary_df["класс"].eq(label)]
    axes[0].hist(subset["char_len"].clip(upper=1500), bins=60, alpha=.55,
                 density=True, label=label, color=color)
    axes[1].hist(subset["word_len"].clip(upper=250), bins=50, alpha=.55,
                 density=True, label=label, color=color)
axes[0].set(title="Символы по классам (1500+ объединены)", xlabel="Символов", ylabel="Плотность")
axes[1].set(title="Слова по классам (250+ объединены)", xlabel="Слов", ylabel="Плотность")
for ax in axes: ax.legend(); ax.grid(axis="y", alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "geo_lengths_by_sentiment.png", dpi=150)
plt.show()

## 8. Выборочный аудит качества слабых меток

Ниже не автоматическая «чистка», а поиск кандидатов по словам с последующим чтением. Ключевые слова дают ложные срабатывания (`неплохой`, «без обмана») и не умеют надёжно распознавать сарказм или смешанную оценку. Поэтому количество найденных строк **не является количеством ошибочных меток**.

In [ ]:
low_positive_pattern = r"садик отличный|отличный и грамотный персонал|отличный магазин,доступные цены|красота все понравилось"
high_negative_pattern = r"хирургу грубиян|брюнетка с очками грубиянка|она не врач"
low_positive_examples = df.loc[
    negative_mask & text.str.contains(low_positive_pattern, case=False, regex=True),
    ["rating", "name_ru", "text"]].head(6)
high_negative_examples = df.loc[
    positive_mask & text.str.contains(high_negative_pattern, case=False, regex=True),
    ["rating", "name_ru", "text"]].head(6)
print("Низкий рейтинг при внешне положительном тексте — кандидаты на несоответствие:")
display(low_positive_examples)
print("Высокий рейтинг при внешне отрицательном тексте — кандидаты на несоответствие:")
display(high_negative_examples)

print("Самые короткие отзывы: формально непустые, но часть из них малоинформативна")
display(df.nsmallest(10, "char_len")[["rating", "text", "char_len"]])
print("Рейтинг 0 встречается, но его смысл не документирован самим значением — примеры:")
display(df.loc[df["rating"].eq(0), ["name_ru", "rubrics", "text"]].head(5))

## 9. Распределение категорий организаций

In [ ]:
category_source = df[["rating", "rubrics", "char_len"]].copy()
category_source["binary_class"] = np.select(
    [category_source["rating"].isin([4, 5]), category_source["rating"].isin([1, 2])],
    ["positive", "negative"], default="excluded")
categories = category_source.assign(rubric=category_source["rubrics"].str.split(";")).explode("rubric")
categories["rubric"] = categories["rubric"].str.strip()
category_counts = categories["rubric"].value_counts()
print("Уникальных отдельных категорий:", category_counts.size)
display(category_counts.head(20).rename("упоминаний").to_frame())

common = categories[categories["binary_class"].ne("excluded")].groupby("rubric").agg(
    отзывов=("rating", "size"),
    средний_рейтинг=("rating", "mean"),
    доля_положительных=("binary_class", lambda s: s.eq("positive").mean()),
    медиана_символов=("char_len", "median"),
).query("отзывов >= 500").sort_values("отзывов", ascending=False).head(20)
display(common.round(3))

fig, ax = plt.subplots(figsize=(10, 6))
category_counts.head(15).sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set(title="15 наиболее частых категорий организаций", xlabel="Упоминаний", ylabel="Категория")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "geo_top_categories.png", dpi=150)
plt.show()

## 10. Ограничения и рекомендации по подготовке данных

- Для первого управляемого эксперимента рациональна стратифицированная рабочая выборка **2 400 отзывов: 1 200 отрицательных и 1 200 положительных**. Это снимает исходный перекос 9,30:1 и оставляет объём, пригодный для двойного ручного аудита части данных.
- Рейтинги 0 и 3 пока хранить отдельно. Не считать `0` автоматически отрицательным без документации источника.
- До разбиения нормализовать только технические пробелы, затем группировать одинаковые тексты. Все копии одного текста должны попадать только в одну из частей train/validation/test.
- При отборе сохранить разнообразие категорий и длины; включить короткие, длинные и смешанные отзывы, а не только самые простые случаи.
- Рейтинг не включать в признаки. Для контрольной части провести независимую двойную ручную оценку текста и фиксировать спорные случаи в codebook.
- Для ручного обогащения целенаправленно искать недостаточно представленный отрицательный класс, но не менять исходный Parquet.

In [ ]:
print("ИТОГОВАЯ СВОДКА")
print("Исходных строк / столбцов:", (len(df), len(SOURCE_COLUMNS)))
print("Пропуски в исходных столбцах:", df[SOURCE_COLUMNS].isna().sum().to_dict())
print("Полные дубликаты (лишние копии):", full_duplicate_extra)
print("Повторяющиеся тексты (лишние копии):", repeated_text_extra)
print("Положительных / отрицательных:", int(positive_mask.sum()), "/", int(negative_mask.sum()))
print("Исключались бы (rating 0 или 3):", excluded)
print("Средняя / медианная длина, символы:", round(df.char_len.mean(), 2), "/", int(df.char_len.median()))
print("Уникальных категорий после разделения по ';':", category_counts.size)
print(f"Время выполнения внутри ноутбука: {perf_counter() - STARTED:.2f} с")